# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2 Croissant dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Install mlcroissant. Uncomment if not already installed.
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print basic dataset metadata
print(f"Dataset name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Identifier: {dataset.metadata.identifier}\nKeywords: {dataset.metadata.keywords}")

## 2. Data Overview
Explore available record sets and their fields/columns, referencing entities by their `@id`.

In [ ]:
# Find all record sets by @id and summarize their fields, referencing by @id
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
record_set_ids = []
for rs in record_sets:
    print(f"- RecordSet '@id': {rs.id}")
    print(f"  Name: {rs.name}")
    record_set_ids.append(rs.id)
    print("  Fields & columns (by @id):")
    for field in rs.fields:
        print(f"    Field @id: {field.id} (name: {field.name}, type: {getattr(field, 'data_type', 'N/A')})")
    for column in rs.columns:
        print(f"    Column @id: {column.id} (name: {column.name}, type: {getattr(column, 'data_type', 'N/A')})")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field/column `@id`s.

In [ ]:
# Collect data into pandas dataframes by record set @id
dfs = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame.from_records(records)
        dfs[rs_id] = df
        print(f"\n{rs_id}: DataFrame shape: {df.shape}")
        print("Columns [@id]:", df.columns.tolist())
        display(df.head(3))

## 4. Exploratory Data Analysis (EDA)
Demonstrate common processing steps, such as filtering records, normalizing numeric fields, or grouping. Use field/column `@id` for all references.

Below we'll demonstrate EDA using the main record set found earlier. Adjust `record_set_id`, `numeric_field_id`, and `group_field_id` to those appropriate for your dataset by `@id`.

In [ ]:
# Example (Replace these @ids with actual ones from the printed outputs above)
record_set_id = record_set_ids[0] if record_set_ids else None  # use the first record set found

if record_set_id is not None and record_set_id in dfs:
    df = dfs[record_set_id]
    # Try to find a numeric field to use (e.g., 'cr:field_coverage_percent', 'cr:field_abundance', or similar)
    # For this dataset, adjust the following IDs after reviewing the earlier output
    # E.g., numeric_field_id = '@id' for a numeric field
    numeric_candidates = [col for col in df.columns if df[col].dtype.kind in 'fi']
    print("Numeric field candidates:", numeric_candidates)
    numeric_field_id = numeric_candidates[0] if numeric_candidates else None
    
    if numeric_field_id is not None:
        # Filtering example
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records in '{record_set_id}' where field '{numeric_field_id}' > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization example
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nFirst few values of normalized '{numeric_field_id}':")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping example (find string/categorical fields to group on)
        group_field_candidates = [col for col in df.columns if df[col].dtype == 'O' and col != numeric_field_id]
        print("Candidates for grouping:", group_field_candidates)
        group_field_id = group_field_candidates[0] if group_field_candidates else None
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nAverage '{numeric_field_id}' grouped by '{group_field_id}':")
            display(grouped_df.head())
    else:
        print("No suitable numeric field found for analysis in this record set.")
else:
    print("No valid record set or data found.")

## 5. Visualization
Here, visualize the main numeric field's distribution and a group comparison if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id is not None and record_set_id in dfs and numeric_field_id is not None:
    df = dfs[record_set_id]
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    # Boxplot by group
    if 'group_field_id' in locals() and group_field_id is not None:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrates loading and analyzing the FAIR^2 dataset using `mlcroissant`.

- We listed available record sets, all fields and columns by `@id`.
- We loaded records into pandas dataframes, filtered and normalized numeric data, and visualized distributions.
- All references to dataset schema components were done using their `@id`, ensuring reproducibility.

Modify field and record set IDs to dig further into other aspects of the dataset, or expand the EDA as needed for your scientific questions.